# 🟠 Galileo — Tier 1: Multi-turn conversation

Three turns sharing one Galileo session. Galileo's auto-metric **Conversation Quality** is designed for exactly this — it scores multi-turn coherence by default.

**Conversation**:
1. "Tell me about Jane Doe — who is she at the company?"
2. "What kind of access does she currently have?"
3. "Given her role, is it appropriate for her to have admin on billing-prod?"

**What to look for in the Galileo UI**:
- Project → log stream `default` → filter session `tier1-multi-turn-galileo`
- The Conversation Quality metric on the session (auto-computed)
- Compare with LangSmith Threads (chat-UI render) and Langfuse Sessions (list render) — same data, three presentations

In [ ]:
import os, sys, pathlib
from dotenv import load_dotenv

ROOT = pathlib.Path().resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env")

os.environ.setdefault("GALILEO_PROJECT", "observability-comparison")
os.environ.setdefault("GALILEO_LOG_STREAM", "default")

from galileo.handlers.langchain import GalileoCallback
from langchain_core.messages import HumanMessage
from shared.workflow import build_agent, MULTI_TURN_CONVERSATION

handler = GalileoCallback()
agent = build_agent(prompt_source="galileo")

SESSION_ID = "tier1-multi-turn-galileo"
print(f"session_id: {SESSION_ID}")
for i, turn in enumerate(MULTI_TURN_CONVERSATION, 1):
    print(f"  Turn {i}: {turn}")

In [ ]:
messages = []
for turn_idx, user_turn in enumerate(MULTI_TURN_CONVERSATION, 1):
    messages.append(HumanMessage(content=user_turn))
    config = {
        "callbacks": [handler],
        "run_name": f"turn_{turn_idx}",
        "tags": ["tier1", "multi_turn", "galileo", f"turn_{turn_idx}"],
        "metadata": {
            "session_id": SESSION_ID,
            "thread_id": SESSION_ID,
            "turn": turn_idx,
        },
    }
    out = agent.invoke({"messages": messages}, config=config)
    messages = out["messages"]

    final = messages[-1].content
    if isinstance(final, list):
        final = " ".join(p.get("text", "") for p in final if isinstance(p, dict))
    print(f"\n--- Turn {turn_idx} ---")
    print(f"User : {user_turn}")
    print(f"Agent: {final[:300]}")

for attr in ("flush", "_flush", "close"):
    fn = getattr(handler, attr, None)
    if callable(fn):
        try:
            fn()
            break
        except Exception:
            pass
import time; time.sleep(3)
print(f"\nDone. Open Galileo, filter session `{SESSION_ID}`. Look at the Conversation Quality metric.")

## Deck takeaway

**Galileo on multi-turn**: gets the **Conversation Quality** auto-metric for free. Whether the agent stayed on topic, whether it answered each turn, whether context was used coherently — all surfaced without writing eval code.

Trade-off: Galileo's Sessions view doesn't render as a chat UI like LangSmith Threads. So you pick: opinionated automatic scoring (Galileo) vs. chat-style visual debugging (LangSmith). For some teams that's a clear winner; for others it's worth running both.